# Stock Exchange Language Model — train on Colab

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run each cell from top to bottom (Shift + Enter).

Training takes about 10–20 minutes on a T4.

In [ ]:
# Check that a GPU is attached (should print a T4).
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Set `REPO_URL` to your GitHub repo. If you haven't pushed it yet, leave it empty and you'll be asked to upload the project zip instead.

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<your-username>/stock-exchange-lm"

import os
if REPO_URL:
    !git clone -q {REPO_URL} project
else:
    from google.colab import files
    up = files.upload()                       # pick stock-exchange-lm.zip
    zip_name = next(iter(up))
    !unzip -q -o "{zip_name}" -d /content/unzipped
    inner = [d for d in os.listdir('/content/unzipped') if not d.startswith('__')]
    !rm -rf project && mv "/content/unzipped/{inner[0]}" project
%cd /content/project
!ls

In [ ]:
# TensorFlow, Flask, datasets and tokenizers are mostly pre-installed on Colab; this fills any gaps.
!pip install -q -r requirements.txt

## 2. Train
This downloads Financial PhraseBank from HuggingFace, trains the BPE tokenizer, then trains the transformer.
Every 5 epochs it prints sample text so you can watch it improve. Early stopping ends training when validation loss stops improving.

In [ ]:
!python train.py

## 3. Try generation

In [ ]:
!python generate.py --prompt "Operating profit rose" --max-tokens 60
!python generate.py --prompt "Shares of the company" --max-tokens 60 --temperature 0.6

## 4. Open the web app inside Colab
This starts Flask in the background and gives you a link that opens the dashboard (live generation + Chart.js training charts).

In [ ]:
import subprocess, time
server = subprocess.Popen(["python", "app.py"], stdout=open("app.log", "w"), stderr=subprocess.STDOUT)
time.sleep(15)   # give TensorFlow time to load
from google.colab.output import eval_js
print("Open this link:", eval_js("google.colab.kernel.proxyPort(5000)"))

## 5. Download the trained model
Unzip this into your project's `checkpoints/` folder on your laptop, then commit it to GitHub so anyone can run the demo without training.

In [ ]:
!zip -q -r checkpoints.zip checkpoints
from google.colab import files
files.download("checkpoints.zip")